# 260515 Self RAG 구현 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w9_agent_rag/llm_260515_self_rag_2.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: Self-RAG 풀 루프 완성 (환각/유용성 + 안전장치)

- **어제와 오늘의 연결**: 어제 retrieve → grade_relevance → (no면 rewrite_query 루프) → generate → grade_hallucination까지 만들었고, 오늘은 그 뒤에 **재생성 루프**를 conditional_edge로 닫고 usefulness 평가까지 붙여 **풀 Self-RAG 그래프**를 완성.
- **할루시네이션 루프의 함정**: 같은 문서·같은 프롬프트로 재생성하면 결과가 안 변함. 두 가지 우회 → ① `temperature=0.7`로 재초기화한 새 LLM으로 regenerate, ② "이전 답변은 환각이었습니다. 문서에 의존해 다시 답변" 같은 **강한 시스템 프롬프트** 추가.
- **환각 vs 유용성은 다른 축**: 환각은 "문서에 근거?"(yes/no), 유용성은 "질문에 정말 답하나?"(1~5점). 근거가 있어도 질문과 동떨어진 답이면 환각 아님이지만 유용성에서 잡힘 → **두 grader 모두 필요**.
- **점프 라우팅(jump routing)**: 5점→END / 3~4점→generate(재생성) / 1~2점→retrieve(처음부터). 그래프 어디로든 점프 가능, 선생님이 강조한 패턴.
- **gpt-4o-mini의 한계**: 토이 예제에선 환각이 잘 안 일어나서 hallucination 루프 검증이 어려움. 데이터가 많아져야 의미 있음 → 일단 로직만 검증하고 넘어감.
- **변별력은 따로 확인**: usefulness grader가 1~5점을 잘 매기는지 `good/bad/unrelated` 답변 세트로 강제 입력해 점검. 좋은 답 3, 나쁜/무관 1 정도 나오면 OK.
- **흐름부터, 코드는 나중**: 라우터·노드를 코드로 바로 적지 말고 "어디→어디"를 먼저 그릴 것. LangGraph 노드 하나하나가 **작은 체인**이라서 `node.invoke({...})`로 떼서 테스트 가능.

## 강의 메모: 실무 팁 (무한 루프 방지 + 디버깅 + C-RAG 맛보기)

- **카운터 통합**: 어제 `relevance_retries`, 오늘 `generation_retries`를 따로 두다가 **`max_retries` 통합 스테이트**로 리팩토링(`safe_relevance/hallucination/usefulness_router`). 모든 conditional_edge의 필수 안전장치.
- **rewrite도 카운팅 대상**: `rewrite_query`가 호출될 때마다 `relevance_retries += 1` 해야 검색 실패 루프가 무한히 안 돌음. 카운터 증가 책임을 어느 노드가 질지 명확히 할 것.
- **stream이 디버깅 생명줄**: `app.stream(...)`으로 노드 단위 출력을 보면 어디서 환각/저점수가 나오는지 즉시 확인. 그래프가 꼬일수록 mermaid 시각화만으론 부족 → stream + mermaid 병행.
- **mermaid 점선 vs 실선**: 실선은 고정 edge, **점선은 conditional_edge**. 분기가 많아질수록 점선 천지라 흐름 추적이 어려워짐.
- **C-RAG의 핵심 차이**: Self-RAG는 yes/no 이진, **C-RAG는 `correct / ambiguous / incorrect` 3단계 + 0~1 confidence score**. ambiguous면 외부 검색 보강으로 분기.
- **knowledge refinement**: 청크를 그대로 쓰지 않고 **문장 단위 분해 → 문장별 관련성 평가 → 관련 문장만 다시 합쳐 정제된 컨텍스트** 생성. "당연한 거 아닌가 싶은 아이디어"라도 논문 되는 분야이니 **떠오르면 일단 적용**해볼 것.